# Assignment 5: 모델 배포 CI/CD 파이프라인 생성
이 과제에서는 모델 배포 파이프라인을 구현합니다.

ML 모델을 프로덕션에 배포하는 방법에는 여러 가지 가능한 패턴이 있습니다. 이러한 패턴은 예를 들어 다중 계정 구조 사용, 테스트 및 승인 워크플로, 환경 단계의 수와 유형과 같은 환경에 따라 달라집니다. 이 실습에서는 간단한 단일 계정 모델 배포를 구현합니다. 그러나 자동화된 CI/CD 프로세스, 모델 단위 테스트, 테스트 및 프로덕션 단계가 있는 2단계 배포, 프로덕션 배포 전 추가 승인과 같은 실제 배포 파이프라인의 모든 주요 요소를 포함하고 있습니다.

모든 필수 인프라와 구성 요소를 AWS 계정에 프로비저닝하기 위해 내장된 [MLOps Project 템플릿](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates.html)을 다시 사용합니다.

다음 다이어그램은 모델 빌드 및 모델 배포 파이프라인으로 구성된 통합 CI/CD 파이프라인의 엔드 투 엔드 아키텍처를 보여줍니다.

![](../img/cicd-project-e2e.png)

다이어그램의 왼쪽 부분은 이전 노트북에서 프로비저닝하고 구성한 **모델 빌드 파이프라인**입니다. 이 노트북에서는 오른쪽 부분인 **모델 배포 파이프라인**을 프로비저닝할 것입니다. 이 두 파이프라인은 모델 레지스트리를 통해 연결됩니다. 모델 빌드 파이프라인은 파이프라인이 성공적으로 실행된 후 모델의 새 버전을 등록합니다. 모델 배포 파이프라인을 시작하려면 모델 레지스트리에서 모델 버전을 승인해야 합니다. 모델 배포 파이프라인은 먼저 승인된 모델 버전을 스테이징 환경에 배포하고, 모델 단위 테스트를 실행하고, 수동 승인을 기다립니다. CI/CD 파이프라인의 **DeployStaging** 단계를 승인하면 파이프라인이 계속되어 모델을 프로덕션 환경에 배포합니다.

모델 배포 프로젝트는 모델 배포를 위해 [AWS Cloud Formation](https://aws.amazon.com/cloudformation/) 템플릿을 사용하여 Infrastructure as Code(IaC) 접근 방식을 따릅니다. 이 프로젝트는 추론 엔드포인트 구성 및 모델 단위 테스트 코드가 있는 소스 코드 리포지토리도 배포합니다.

이 과제의 연습을 위한 코드 스니펫과 일반적인 지침은 [`05-deploy.ipynb`](../05-deploy.ipynb) 노트북을 참조하십시오.

## 패키지 임포트

In [ ]:
import boto3
import sagemaker 
from time import gmtime, strftime, sleep

## 연습 1: 모델 배포 프로젝트 생성
Python SDK `boto3`를 사용하여 프로그래밍 방식으로 프로젝트를 생성하려면 [`05-deploy.ipynb`](../05-deploy.ipynb) 노트북의 코드를 재사용할 수 있습니다.

또는 Studio UX를 통해 새 프로젝트를 프로비저닝할 수 있습니다.

In [ ]:
# 프로젝트를 생성한 후 프로젝트 세부 정보 가져오기
# project_name = <YOUR PROJECT NAME>
# sm.describe_project(ProjectName=project_name)

## 연습 2: 프로젝트 구성 요소 탐색
SageMaker MLOps 프로젝트는 AWS 계정에 엔드 투 엔드 CI/CD 모델 배포 파이프라인을 프로비저닝합니다:

![](../img/mlops-model-deploy-2.png)

주요 구성 요소는 다음과 같습니다:
1. 구성, 테스트 및 워크플로 코드가 있는 AWS CodeCommit 리포지토리
2. 4개의 단계가 있는 AWS CodePipeline 배포 파이프라인
3. 모델 패키지 버전이 승인되거나 거부될 때 CodePipeline 파이프라인 실행을 시작하는 Amazon EventBridge 규칙
4. 프로덕션 추론 엔드포인트를 배포하기 위한 모델 단위 테스트 후 수동 승인 단계

AWS 콘솔에서 각 파이프라인 구성 요소를 탐색하십시오.

### AWS CodeCommit 리포지토리
프로젝트 코드 리포지토리를 Studio 홈 디렉터리에 [복제](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-walkthrough.html#sagemaker-proejcts-walkthrough-clone)하고 소스 코드를 탐색하십시오. Studio 파일 시스템의 프로젝트 폴더로 이동하십시오.

`README.md` 파일은 리포지토리 구조와 각 파일에 대한 설명을 자세히 제공합니다. 소스 코드의 주요 구성 요소:
- 각각 자체 `buildspec.yml` 파일이 있는 두 개의 빌드 단계: 최신 승인된 모델 패키지를 검색하고 구성 파일을 빌드하는 단계, 그리고 단위 테스트를 위한 단계
- 각 빌드 단계에 대한 실행 가능한 코드가 있는 두 개의 Python 스크립트 `build.py` 및 `test/test.py`
- 실시간 추론 엔드포인트로서 IaC 모델 배포를 위한 CloudFormation 템플릿 `endpoint-config-template.yml`
- 두 개의 구성 파일 `staging-config.json` 및 `prod-config.json`. 파이프라인의 배포 단계는 이러한 파일을 사용하여 컴퓨팅 인스턴스 유형, 데이터 캡처 구성, 태그와 같은 추론 엔드포인트의 매개변수를 구성합니다

MLOps 프로젝트는 자동화된 배포 파이프라인에 대한 가능한 접근 방식 중 하나만 구현합니다. 특정 환경에 따라 다른 기술 스택을 사용하여 다른 접근 방식을 구현할 수 있습니다. 여전히 SageMaker 프로젝트 템플릿을 사용하여 사전 정의된 CI/CD 파이프라인 템플릿을 생성하기 위해 모든 구성 요소와 기술 스택을 패키징할 수 있습니다.

### AWS CodePipeline 파이프라인
[AWS CodePipeline 콘솔](https://console.aws.amazon.com/codesuite/codepipeline/pipelines)을 열고 `sagemaker-<project-name>-<project-id>-modeldeploy`라는 이름의 모델 배포 파이프라인을 탐색하십시오. 파이프라인에는 4개의 단계가 있습니다. 단계 간 전환과 각 단계에서 생성된 아티팩트를 탐색하십시오.

예를 들어 빌드 단계를 탐색하십시오. [AWS CodeBuild 콘솔](https://console.aws.amazon.com/codesuite/codebuild/projects)로 이동하여 `sagemaker-<project-name>-<project-id>-modeldeploy`라는 이름의 빌드 프로젝트를 열고 **Environment variables** 패널을 여십시오. MLOps 프로젝트 템플릿에 의해 설정된 환경 변수를 볼 수 있습니다:

![](../img/codebuild-env-variables.png)

프로젝트 폴더에서 `build.py` 파일을 여십시오. 실행 흐름과 구현이 전달된 인수를 사용하여 구성 매개변수를 설정하고 구성 파일을 생성하는 방법을 탐색하십시오.

CodePipeline 파이프라인의 **DeployStaging** 단계를 탐색하십시오. 파이프라인 목록에서 파이프라인 이름을 클릭한 다음 **DeployStaging** 단계의 **DeployResourceStaging** 액션 옆에 있는 "i" 아이콘을 클릭하십시오. CloudFormation 배포 액션이 구성된 방법을 볼 수 있습니다:

![](../img/deploystaging-config.png)

프로젝트 폴더에서 `/test/test.py` 파일을 여십시오. 엔드포인트 테스트가 어떻게 수행되는지 탐색하십시오. `invoke_endpoint` 함수를 구현하고 페이로드를 엔드포인트로 전송하여 테스트할 수 있습니다.

수동 승인, 프로덕션 엔드포인트 배포와 같은 파이프라인의 다른 단계를 계속 탐색하십시오.

### Amazon EventBridge 규칙
EventBridge [콘솔](https://console.aws.amazon.com/events/home?#/rules)의 규칙으로 이동하여 `sagemaker-<project-name>-<project-id>-code` 및 `sagemaker-<project-name>-<project-id>-model`이라는 이름의 규칙을 찾으십시오. 첫 번째 규칙 `...-code`는 소스 코드 리포지토리로의 각 푸시에서 파이프라인을 시작합니다. 두 번째 규칙 `...-model`은 모든 모델 패키지 상태 변경 시 파이프라인을 시작합니다. **Event pattern**을 참조하십시오:

```
{
  "detail-type": ["SageMaker Model Package State Change"],
  "source": ["aws.sagemaker"],
  "detail": {
    "ModelPackageGroupName": ["from-idea-to-prod-model-group"]
  }
}
```

## 연습 3: 모델 배포 파이프라인 실행
배포 파이프라인을 시작하려면 모델 레지스트리에서 모델 버전을 승인해야 합니다.

Studio UX의 모델 레지스트리에서 모델 버전을 승인하거나 노트북에서 프로그래밍 방식으로 승인할 수 있습니다.

[AWS CodePipeline 콘솔](https://console.aws.amazon.com/codesuite/codepipeline/pipelines)에서 파이프라인 실행을 따르십시오. 파이프라인이 스테이징 추론 엔드포인트를 배포한 후 프로덕션 엔드포인트 배포를 수동으로 승인하십시오. 지침은 [`05-deploy.ipynb`](../05-deploy.ipynb) 노트북을 참조하십시오.

배포 파이프라인이 성공적으로 완료된 후 두 개의 `InService` 엔드포인트가 있는지 확인하십시오:

<img src="../img/endpoint-prod.png" width="400"/>

## Assignment 6 계속하기
[assignment 6](06-assignment-monitoring.ipynb) 노트북으로 이동합니다.